In [8]:
import cv2
from IPython.display import display, clear_output
from ultralytics import YOLO
import time

In [11]:
# Charger YOLOv8n en mode tracking
model = YOLO('yolov8n.pt')  # ou 'yolov8n.yaml' pour un modèle non pré-entraîné

## Via YOLO

In [12]:
# Vidéo locale à analyser
video_path = "853874-hd_1920_1080_25fps.mp4"
cap = cv2.VideoCapture(video_path)

# Affichage dans le notebook
from IPython.display import Image, display
import PIL.Image
import io

def show_frame(frame):
    _, img_encoded = cv2.imencode('.jpg', frame)
    display(PIL.Image.open(io.BytesIO(img_encoded)))

# Boucle de détection + tracking
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # YOLOv8 avec suivi automatique (track=True)
    results = model.track(frame, persist=True, verbose=False)
    
    # Annoter les résultats sur l'image
    annotated_frame = results[0].plot()

    # Affichage dans le notebook
    clear_output(wait=True)
    show_frame(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))

    # Pour ralentir un peu la lecture
    #time.sleep(0.03)

cap.release()


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Defaulting to user installation because normal site-packages is not writeable

requirements: AutoUpdate success ✅ 1.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



ModuleNotFoundError: No module named 'lap'

## Via deepSORT

In [13]:
from ultralytics import YOLO
import cv2
from deep_sort_realtime.deepsort_tracker import DeepSort
from IPython.display import display, clear_output
from PIL import Image
import io
import time

# Init modèles
model = YOLO('yolov8n.pt')  # ou yolov8s.pt selon ta machine
tracker = DeepSort(max_age=30)

# Vidéo locale
video_path = "853874-hd_1920_1080_25fps.mp4"
cap = cv2.VideoCapture(video_path)

#start = time.time()

def show_frame_inline(frame):
    _, buffer = cv2.imencode('.jpg', frame)
    img = Image.open(io.BytesIO(buffer))
    clear_output(wait=True)
    display(img)

# Lecture vidéo + tracking
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    detections = []
    for box in results.boxes.data:
        x1, y1, x2, y2, conf, cls = box.tolist()
        detections.append((
            [x1, y1, x2 - x1, y2 - y1],
            conf,
            model.names[int(cls)]
        ))

    tracks = tracker.update_tracks(detections, frame=frame)
    
    for track in tracks:
        if not track.is_confirmed():
            continue
        l, t, r, b = track.to_ltrb()
        track_id = track.track_id
        class_name = track.get_det_class() or "obj"

        label = f'{class_name} #{track_id}'
        cv2.rectangle(frame, (int(l), int(t)), (int(r), int(b)), (0, 255, 0), 2)
        cv2.putText(frame, label, (int(l), int(t) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    #end = time.time()
    #fps = 1 / (end - start)
    #cv2.putText(frame, f'FPS: {fps:.2f}', (20, 30),
    #            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)


    # Affichage direct (rapide)
    #cv2.imshow("YOLOv8 + DeepSORT", frame)
    #if cv2.waitKey(1) & 0xFF == ord('q'):
    #    break

    # Conversion BGR → RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Affichage inline
    show_frame_inline(frame_rgb)
    time.sleep(0.03)  # ralentir un peu la lecture (30 fps)

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

## Flux video streaming

In [29]:
import subprocess

# Paramètres du stream (par exemple, sur localhost:1234)
udp_address = "udp://127.0.0.1:1234"

# Commande FFmpeg (encode H264, envoie sur UDP)
ffmpeg_cmd = [
    'ffmpeg',
    '-y',
    '-f', 'rawvideo',
    '-vcodec', 'rawvideo',
    '-pix_fmt', 'bgr24',
    '-s', f'{int(cap.get(3))}x{int(cap.get(4))}',
    '-r', '30',
    '-i', '-',  # stdin
    '-an',
    '-c:v', 'libx264',
    '-preset', 'ultrafast',
    '-f', 'mpegts',
    udp_address
]

# Démarre le process ffmpeg
ffmpeg_process = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)


ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-libtwolame --enable-libvidstab --enab

In [30]:
from ultralytics import YOLO
import cv2
from deep_sort_realtime.deepsort_tracker import DeepSort
from IPython.display import display, clear_output
from PIL import Image
import io
import time

# Init modèles
model = YOLO('yolov8n.pt')  # ou yolov8s.pt selon ta machine
tracker = DeepSort(max_age=30)

# Vidéo locale
video_path = "853874-hd_1920_1080_25fps.mp4"
cap = cv2.VideoCapture(video_path)

#start = time.time()

def show_frame_inline(frame):
    _, buffer = cv2.imencode('.jpg', frame)
    img = Image.open(io.BytesIO(buffer))
    clear_output(wait=True)
    display(img)

# Lecture vidéo + tracking
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    detections = []
    for box in results.boxes.data:
        x1, y1, x2, y2, conf, cls = box.tolist()
        detections.append((
            [x1, y1, x2 - x1, y2 - y1],
            conf,
            model.names[int(cls)]
        ))

    tracks = tracker.update_tracks(detections, frame=frame)
    
    for track in tracks:
        if not track.is_confirmed():
            continue
        l, t, r, b = track.to_ltrb()
        track_id = track.track_id
        class_name = track.get_det_class() or "obj"

        label = f'{class_name} #{track_id}'
        cv2.rectangle(frame, (int(l), int(t)), (int(r), int(b)), (0, 255, 0), 2)
        cv2.putText(frame, label, (int(l), int(t) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        ffmpeg_process.stdin.write(frame.tobytes())

cap.release()
cv2.destroyAllWindows()
ffmpeg_process.stdin.close()
ffmpeg_process.wait()


0: 384x640 15 persons, 1 train, 1 backpack, 1 handbag, 27.6ms
Speed: 11.1ms preprocess, 27.6ms inference, 10.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 persons, 1 backpack, 1 handbag, 23.0ms
Speed: 3.0ms preprocess, 23.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 2 backpacks, 2 handbags, 28.5ms
Speed: 3.3ms preprocess, 28.5ms inference, 4.2ms postprocess per image at shape (1, 3, 384, 640)


Input #0, rawvideo, from 'pipe:':
  Duration: N/A, start: 0.000000, bitrate: 1492992 kb/s
  Stream #0:0: Video: rawvideo (BGR[24] / 0x18524742), bgr24, 1920x1080, 1492992 kb/s, 30 tbr, 30 tbn
Stream mapping:
  Stream #0:0 -> #0:0 (rawvideo (native) -> h264 (libx264))
[libx264 @ 0x5d509c251540] using cpu capabilities: MMX2 SSE2Fast SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
[libx264 @ 0x5d509c251540] profile High 4:4:4 Predictive, level 4.0, 4:4:4, 8-bit
Output #0, mpegts, to 'udp://127.0.0.1:1234':
  Metadata:
    encoder         : Lavf59.27.100
  Stream #0:0: Video: h264, yuv444p(tv, progressive), 1920x1080, q=2-31, 30 fps, 90k tbn
    Metadata:
      encoder         : Lavc59.37.100 libx264
    Side data:
      cpb: bitrate max/min/avg: 0/0/0 buffer size: 0 vbv_delay: N/A
frame=    1 fps=0.0 q=0.0 size=       0kB time=00:00:00.00 bitrate=N/A speed=N/A    


0: 384x640 14 persons, 2 backpacks, 13.2ms
Speed: 3.6ms preprocess, 13.2ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)


frame=   18 fps=0.0 q=15.0 size=     296kB time=00:00:00.13 bitrate=18159.3kbits/s speed=0.238x    


0: 384x640 14 persons, 3 backpacks, 41.7ms
Speed: 13.1ms preprocess, 41.7ms inference, 5.9ms postprocess per image at shape (1, 3, 384, 640)


...


0: 384x640 17 persons, 1 train, 2 backpacks, 24.4ms
Speed: 4.7ms preprocess, 24.4ms inference, 6.1ms postprocess per image at shape (1, 3, 384, 640)


frame= 3199 fps= 37 q=15.0 size=   63504kB time=00:01:46.16 bitrate=4900.1kbits/s speed=1.24x    


0: 384x640 13 persons, 1 train, 1 backpack, 14.3ms
Speed: 3.6ms preprocess, 14.3ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)


frame= 3236 fps= 37 q=16.0 size=   64177kB time=00:01:47.40 bitrate=4895.2kbits/s speed=1.24x    

KeyboardInterrupt: 